# Segmented prime sieve

Count primes in a large integer range by splitting it into segments and sieving each segment on a worker. The serial-vs-parallel timing at the end gives a feel for the speedup on your cluster.

The task body is pure Python standard library -- no numpy, no sklearn, no extra wheels. The result of each task is a single integer (the count of primes in the segment), so the network bandwidth stays trivial regardless of how big the search range gets.

**Prerequisites**: a Scaler scheduler reachable from this kernel. See :doc:`../tutorials/try_in_browser` for the recommended worker setup.

In [ ]:
# Connection settings -- edit these to point at your running cluster.
SCHEDULER_ADDRESS = "ws://127.0.0.1:2345"  # supports tcp:// or ws://; only ws:// works from JupyterLite (browser)
OBJECT_STORAGE_ADDRESS = None  # leave None to use whatever the scheduler advertises

SEARCH_UPPER = 20_000_000   # count primes in [2, SEARCH_UPPER)
N_SEGMENTS = 32             # split into this many worker tasks

In [ ]:
import time

from scaler import Client


def _small_primes(upper: int) -> list[int]:
    """Plain Eratosthenes sieve up to (but not including) `upper`."""
    if upper < 3:
        return []
    flags = bytearray([1]) * upper
    flags[0] = flags[1] = 0
    for i in range(2, int(upper ** 0.5) + 1):
        if flags[i]:
            flags[i * i :: i] = bytearray(len(flags[i * i :: i]))
    return [i for i, flag in enumerate(flags) if flag]


def count_primes_in_segment(low: int, high: int, base_primes: list[int]) -> int:
    """Worker-side: segmented Eratosthenes sieve over [low, high)."""
    if high <= 2:
        return 0
    low = max(low, 2)
    size = high - low
    flags = bytearray([1]) * size
    for p in base_primes:
        if p * p >= high:
            break
        # first multiple of p in [low, high)
        start = max(p * p, ((low + p - 1) // p) * p)
        flags[start - low :: p] = bytearray(len(flags[start - low :: p]))
    return sum(flags)


segment_size = SEARCH_UPPER // N_SEGMENTS
segments = [(i * segment_size, (i + 1) * segment_size if i < N_SEGMENTS - 1 else SEARCH_UPPER) for i in range(N_SEGMENTS)]
base_primes = _small_primes(int(SEARCH_UPPER ** 0.5) + 1)

with Client(address=SCHEDULER_ADDRESS, object_storage_address=OBJECT_STORAGE_ADDRESS) as client:
    base_ref = client.send_object(base_primes, name="base-primes")
    started = time.perf_counter()
    futures = [client.submit(count_primes_in_segment, low, high, base_ref) for low, high in segments]
    parallel_total = sum(f.result() for f in futures)
    parallel_elapsed = time.perf_counter() - started

print(f"parallel:  {parallel_total:,} primes below {SEARCH_UPPER:,} in {parallel_elapsed:.2f}s ({N_SEGMENTS} segments)")

In [ ]:
# Compare against a single-segment baseline run on one worker.
with Client(address=SCHEDULER_ADDRESS, object_storage_address=OBJECT_STORAGE_ADDRESS) as client:
    started = time.perf_counter()
    serial_total = client.submit(count_primes_in_segment, 0, SEARCH_UPPER, base_primes).result()
    serial_elapsed = time.perf_counter() - started

speedup = serial_elapsed / parallel_elapsed if parallel_elapsed > 0 else float("inf")
print(f"serial:    {serial_total:,} primes below {SEARCH_UPPER:,} in {serial_elapsed:.2f}s (single segment)")
print(f"speedup:   {speedup:.2f}x")
assert serial_total == parallel_total